In [31]:
import pandas as pd
import numpy as np

# Настройки для более удобного отображения pandas в Jupyter
pd.set_option('display.max_rows', 200)

# --- Шаг 1: Загрузка данных ---
try:
    bcfdb_df = pd.read_csv('bcfdb.csv')
    llm_df = pd.read_csv('llm_dataset.csv')
    print("Файлы 'bcfdb.csv' и 'llm_dataset.csv' успешно загружены.")
except FileNotFoundError:
    print(f"Ошибка: Файл не найден.")
    assert False, "Остановите выполнение, файлы не найдены"

# Ключ агрегации: усредняем BCF по всем частям растения
grouping_columns = ['doi', 'variety', 'element']

# --- Шаг 2: Очистка и нормализация ---
print("Нормализация данных...")
string_columns_to_clean = ['variety', 'element', 'plant_component']

for df in [bcfdb_df, llm_df]:
    for col in string_columns_to_clean:
        if col in df.columns and pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].str.strip()

# --- Шаг 3: Агрегация данных ---
bcfdb_df['bcf'] = pd.to_numeric(bcfdb_df['bcf'], errors='coerce')
llm_df['bcf'] = pd.to_numeric(llm_df['bcf'], errors='coerce')
bcfdb_df.dropna(subset=['bcf'] + grouping_columns, inplace=True)
llm_df.dropna(subset=['bcf'] + grouping_columns, inplace=True)

print("Агрегация данных (усреднение по частям растения)...")
avg_bcfdb_df = bcfdb_df.groupby(grouping_columns)['bcf'].mean().reset_index()
avg_llm_df = llm_df.groupby(grouping_columns)['bcf'].mean().reset_index()
print("Агрегация завершена.")

# --- Шаг 4: Объединение и расчеты ---
comparison_df = pd.merge(
    avg_bcfdb_df,
    avg_llm_df,
    on=grouping_columns,
    suffixes=('_original', '_llm')
)
print(f"\nНайдено {len(comparison_df)} общих записей для сравнения.")

if len(comparison_df) > 0:
    comparison_df.rename(columns={
        'bcf_original': 'BCF_Original (Mean)',
        'bcf_llm': 'BCF_LLM (Mean)'
    }, inplace=True)

    comparison_df['Abs_Difference'] = (comparison_df['BCF_Original (Mean)'] - comparison_df['BCF_LLM (Mean)']).abs()
    comparison_df['Rel_Difference_%'] = (comparison_df['Abs_Difference'] / comparison_df['BCF_Original (Mean)']) * 100
    comparison_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    comparison_df['Rel_Difference_%'] = comparison_df['Rel_Difference_%'].fillna(0)
    
    # Сортируем по DOI, затем по сорту и элементу
    sorted_comparison_df = comparison_df.sort_values(
        by=['doi', 'variety', 'element'], 
        ascending=[True, True, True]
    ).reset_index(drop=True)
    
    final_columns = [
        'doi', 'variety', 'element', 
        'BCF_Original (Mean)', 'BCF_LLM (Mean)', 'Abs_Difference', 'Rel_Difference_%'
    ]
    final_table = sorted_comparison_df[final_columns]

    # --- Шаг 5: Стилизация и вывод таблицы ---
    
    styled_table = final_table.style.background_gradient(
        # ==================================================================
        # === ВОЗВРАЩАЕМ ФИКСИРОВАННУЮ ШКАЛУ ===
        # Градиент будет растянут на диапазон от 0% до 100%
        cmap='RdYlGn_r',
        subset=['Rel_Difference_%'],
        vmin=0,   # Начало градиента - 0% (самый зеленый)
        vmax=25  # Конец градиента - 100% и выше (самый красный)
        # ==================================================================
    ).format({
        'BCF_Original (Mean)': '{:.4f}',
        'BCF_LLM (Mean)': '{:.4f}',
        'Abs_Difference': '{:.4f}',
        'Rel_Difference_%': '{:.2f}%'
    }).set_caption(
        "<h3>Сравнительная таблица BCF (шкала 0-100%, сортировка по DOI)</h3>"
    ).set_properties(**{'text-align': 'center'})

    display(styled_table)

else:
    print("\nОбщих записей для сравнения не найдено.")

Файлы 'bcfdb.csv' и 'llm_dataset.csv' успешно загружены.
Нормализация данных...
Агрегация данных (усреднение по частям растения)...
Агрегация завершена.

Найдено 34 общих записей для сравнения.


,doi,variety,element,BCF_Original (Mean),BCF_LLM (Mean),Abs_Difference,Rel_Difference_%
0,10.1080/01904167.2021.1881553,Bamahuoma,Pb,0.1308,0.1447,0.0139,10.61%
1,10.1186/1752-153x-6-122,Armanca,Ca,1.1052,1.0100,0.0952,8.62%
2,10.1186/1752-153x-6-122,Armanca,Fe,0.0841,0.0800,0.0041,4.86%
3,10.1186/1752-153x-6-122,Armanca,K,42.7026,40.4400,2.2626,5.30%
4,10.1186/1752-153x-6-122,Armanca,Mg,10.6875,9.8400,0.8475,7.93%
5,10.1186/1752-153x-6-122,Armanca,Mn,0.5192,0.4600,0.0592,11.41%
6,10.1186/1752-153x-6-122,Denise,Ca,0.8381,0.9100,0.0719,8.58%
7,10.1186/1752-153x-6-122,Denise,Fe,0.0820,0.0900,0.0080,9.77%
8,10.1186/1752-153x-6-122,Denise,K,23.6459,24.8000,1.1541,4.88%
9,10.1186/1752-153x-6-122,Denise,Mg,8.3484,9.0400,0.6916,8.28%
